### Итерация Политики для среды CliffWalking-v0

Этот ноутбук демонстрирует алгоритм Итерации Политики, фундаментальный метод в обучении с подкреплением для нахождения оптимальной политики для заданного Марковского процесса принятия решений (МППР). Мы применим его к среде `CliffWalking-v0` из библиотеки `gymnasium`.

**Описание среды (`CliffWalking-v0`):**
Агент начинает с обозначенного состояния 'S' (старт) и стремится достичь состояния 'G' (цель). В среде есть область 'скалы'. Если агент наступает на скалу, он немедленно возвращается в начальное состояние. Перемещение в нескалистое состояние приносит вознаграждение -1. Падение со скалы приносит вознаграждение -100 и сбрасывает в начальное состояние.

### Шаги:
1.  **Установить `gymnasium`:** Библиотека для сред.
2.  **Инициализировать среду:** Загрузить `CliffWalking-v0`.
3.  **Реализовать Оценку Политики:** Вычислить функцию ценности для данной политики.
4.  **Реализовать Улучшение Политики:** Обновить политику на основе функции ценности.
5.  **Реализовать Итерацию Политики:** Итерировать между оценкой и улучшением до сходимости политики.
6.  **Визуализировать результаты:** Отобразить оптимальную политику и функцию ценности.

In [49]:
# Install gymnasium for the environment
!pip install gymnasium


In [50]:
import numpy as np
import gymnasium as gym

# Создание среды CliffWalking-v1 (обновлено с v0)
env = gym.make('CliffWalking-v1')

# Получение количества состояний и действий
n_states = env.observation_space.n
n_actions = env.action_space.n

print(f"Количество состояний: {n_states}")
print(f"Количество действий: {n_actions}")

# Инициализация случайной политики (равновероятная для всех действий)
# policy[s, a] дает вероятность выполнения действия 'a' в состоянии 's'
policy = np.ones((n_states, n_actions)) / n_actions

# Инициализация функции ценности (V) нулями
V = np.zeros(n_states)

print("Начальная политика (первые 5 состояний):\n", policy[:5])
print("Начальная функция ценности (первые 5 состояний):\n", V[:5])

Количество состояний: 48
Количество действий: 4
Начальная политика (первые 5 состояний):
 [[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]
Начальная функция ценности (первые 5 состояний):
 [0. 0. 0. 0. 0.]


### 3. Реализация Оценки Политики (Policy Evaluation)

На этом шаге мы оцениваем, насколько хороша текущая политика, вычисляя функцию ценности состояния $V(s)$ для каждого состояния. Функция ценности $V(s)$ представляет собой ожидаемую сумму дисконтированных вознаграждений, начиная со состояния $s$ и следуя данной политике.

In [51]:
def policy_evaluation(policy, env, gamma=1.0, theta=1e-8):
    """
    Выполняет оценку политики.
    """
    V = np.zeros(env.observation_space.n)
    while True:
        delta = 0
        for s in range(env.observation_space.n):
            # Если это целевое состояние, его ценность всегда 0
            if s == env.observation_space.n - 1:
                continue

            v = 0
            for a, action_prob in enumerate(policy[s]):
                for prob, next_state, reward, terminated in env.unwrapped.P[s][a]:
                    # Уравнение Беллмана: V(s) = sum(pi * p * (r + gamma * V(s')))
                    # В gymnasium CliffWalking terminated=True только на Goal
                    v += action_prob * prob * (reward + gamma * V[next_state])

            delta = max(delta, np.abs(v - V[s]))
            V[s] = v
        if delta < theta:
            break
    return V

### 4. Реализация Улучшения Политики (Policy Improvement)

На этом шаге мы улучшаем текущую политику, делая ее жадной по отношению к функции ценности, вычисленной на предыдущем шаге. Для каждого состояния мы выбираем действие, которое максимизирует ожидаемое вознаграждение, плюс дисконтированное значение следующего состояния. Если несколько действий дают одинаковое максимальное значение, мы можем распределить вероятность между ними.

In [52]:
def policy_improvement(current_policy, env, V, gamma=1.0):
    """
    Улучшает политику на основе функции ценности.
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    new_policy = np.zeros((n_states, n_actions))
    policy_changed = False

    for s in range(n_states):
        if s == n_states - 1: # Пропускаем целевое состояние
            new_policy[s] = current_policy[s]
            continue

        old_actions = np.copy(current_policy[s])
        action_values = np.zeros(n_actions)

        for a in range(n_actions):
            for prob, next_state, reward, terminated in env.unwrapped.P[s][a]:
                action_values[a] += prob * (reward + gamma * V[next_state])

        # Находим лучшие действия (аргмакс с учетом точности float)
        best_value = np.max(action_values)
        best_actions = np.where(np.isclose(action_values, best_value, atol=1e-8))[0]

        new_policy[s, best_actions] = 1.0 / len(best_actions)

        if not np.allclose(old_actions, new_policy[s]):
            policy_changed = True

    return new_policy, not policy_changed

### 5. Реализация Итерации Политики (Policy Iteration)

Итерация Политики объединяет два предыдущих шага: Оценку Политики и Улучшение Политики. Алгоритм итеративно выполняет эти шаги, пока политика не станет стабильной, то есть пока дальнейшее улучшение политики не приведет к ее изменению. В этот момент мы найдем оптимальную политику и соответствующую ей оптимальную функцию ценности.

In [53]:
def policy_iteration(env, gamma=0.9, theta=1e-6):
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    policy = np.ones((n_states, n_actions)) / n_actions

    iterations = 0
    while True:
        iterations += 1
        V = policy_evaluation(policy, env, gamma, theta)
        new_policy, policy_stable = policy_improvement(policy, env, V, gamma)

        if policy_stable:
            break
        policy = new_policy
        print(f"Итерация {iterations}: политика обновлена")

    print(f"Итерация политики завершена за {iterations} итераций.")
    return policy, V

# Перезапуск с чистыми параметрами
optimal_policy, optimal_V = policy_iteration(env, gamma=0.9, theta=1e-6)

print("\nПроверка политики завершена.")

Итерация 1: политика обновлена
Итерация 2: политика обновлена
Итерация 3: политика обновлена
Итерация 4: политика обновлена
Итерация политики завершена за 5 итераций.

Проверка политики завершена.


### 6. Визуализация Результатов

Давайте визуализируем найденную оптимальную политику и функцию ценности. Для среды `CliffWalking` оптимальная политика обычно отображается в виде стрелок, указывающих оптимальное действие в каждом состоянии, а функция ценности - числовым значением в каждом состоянии.

In [54]:
def print_policy(policy, env):
    """
    Визуализирует политику для среды CliffWalking.
    """
    # 0: Up, 1: Right, 2: Down, 3: Left
    actions = ['↑', '→', '↓', '←']

    # CliffWalking-v1 имеет фиксированные размеры 4x12
    nrows = 4
    ncols = 12

    # Форматирование для вывода
    print("\nОптимальная политика:")
    grid_policy = []
    for i in range(nrows):
        row_policy = []
        for j in range(ncols):
            s = i * ncols + j
            if s == (nrows * ncols - 1): # Целевое состояние (последнее состояние)
                row_policy.append('G')
            elif s >= (nrows - 1) * ncols + 1 and s <= (nrows * ncols - 2): # Состояния скалы (ряд 3, колонки 1-10)
                row_policy.append('C')
            else:
                optimal_action = np.argmax(policy[s])
                row_policy.append(actions[optimal_action])
        grid_policy.append(" ".join(row_policy))
    print("\n".join(grid_policy))

def print_value_function(V, env):
    """
    Визуализирует функцию ценности для среды CliffWalking.
    """
    # CliffWalking-v1 имеет фиксированные размеры 4x12
    nrows = 4
    ncols = 12

    print("\nОптимальная функция ценности (округлено до 2 знаков после запятой):")
    grid_values = []
    for i in range(nrows):
        row_values = []
        for j in range(ncols):
            s = i * ncols + j
            if s == (nrows * ncols - 1):
                row_values.append(f"{0.00: 7.2f}") # Ценность цели 0
            elif s >= (nrows - 1) * ncols + 1 and s <= (nrows * ncols - 2):
                row_values.append(f"{'CLIFF':<7s}") # Отметка скалы
            else:
                row_values.append(f"{V[s]: 7.2f}")
        grid_values.append(" ".join(row_values))
    print("\n".join(grid_values))

# Вывод оптимальной политики и функции ценности
print_policy(optimal_policy, env)
print_value_function(optimal_V, env)

# Закрытие среды
env.close()


Оптимальная политика:
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
↑ C C C C C C C C C C G

Оптимальная функция ценности (округлено до 2 знаков после запятой):
  -7.71   -7.46   -7.18   -6.86   -6.51   -6.13   -5.70   -5.22   -4.69   -4.10   -3.44   -2.71
  -7.46   -7.18   -6.86   -6.51   -6.13   -5.70   -5.22   -4.69   -4.10   -3.44   -2.71   -1.90
  -7.18   -6.86   -6.51   -6.13   -5.70   -5.22   -4.69   -4.10   -3.44   -2.71   -1.90   -1.00
  -7.46 CLIFF   CLIFF   CLIFF   CLIFF   CLIFF   CLIFF   CLIFF   CLIFF   CLIFF   CLIFF      0.00


### Интерпретация результатов

Визуализация оптимальной политики и функции ценности демонстрирует, как агент научился перемещаться по среде `CliffWalking-v1`.

*   **Оптимальная политика:** Агент последовательно избегает области скал (помеченных 'C') и выбирает действия, которые ведут его к цели ('G') с минимальными штрафами. В большинстве безопасных состояний на нижней строке (непосредственно перед скалами) агент выбирает действие 'вверх' (↑), чтобы избежать падения. В других состояниях, чтобы максимально быстро достичь нижней строки перед скалами, агент преимущественно движется 'влево' (←) или 'вверх' (↑) в верхних строках и 'вправо' (→) в первой строке.

*   **Оптимальная функция ценности:** Значения функции ценности отражают ожидаемое совокупное вознаграждение, которое агент получит, начиная с каждого состояния и следуя оптимальной политике. Чем ближе состояние к цели и чем безопаснее путь к ней, тем выше (менее отрицательна) ценность состояния. Нулевая ценность в состоянии 'G' подтверждает, что это конечная цель. Очень низкие отрицательные значения в других состояниях показывают накопленный штраф (-1 за каждый шаг) до достижения цели. Состояния 'CLIFF' не имеют числовой ценности, поскольку при попадании в них агент получает большой штраф и возвращается к началу, что уже учтено в вычислениях функции ценности для других состояний.